# Run fMRIPrep on the fMRI Course BIDS dataset

Runs [fMRIPrep](https://fmriprep.org/) via Neurodesk's container module on
the BIDS dataset produced by `bids_conversion.ipynb`:

```
/home/jovyan/Desktop/fmri_course_JLU/bids/
└── sub-2604201/{anat,func,fmap}/
```

Adapted from the Neurodesk tutorial:
<https://neurodesk.org/edu/examples/functional_imaging/fmriprep.html>

## 1. Load the fMRIPrep module

Neurodesk exposes fMRIPrep as a container module. Loading it puts the
`fmriprep` wrapper script on `PATH`.

In [ ]:
import module
await module.load('fmriprep/24.1.0')
await module.list()

## 2. FreeSurfer license

fMRIPrep calls FreeSurfer (for cortical surface reconstruction) and
FreeSurfer refuses to run without a license file. Request a free
license from <https://surfer.nmr.mgh.harvard.edu/registration.html>
(takes a few minutes), then paste the four lines below.

**Replace the placeholder lines with the real license from your email.**

In [ ]:
from pathlib import Path

# PASTE YOUR REAL FREESURFER LICENSE HERE.
# Register at https://surfer.nmr.mgh.harvard.edu/registration.html — free, 5 min.
# You'll receive an email whose body contains four lines; copy them verbatim between
# the triple-quoted markers below.
LICENSE_TEXT = """
COPY AND PASTE YOUR LICENSE INFO HERE
"""

LICENSE_PATH = Path.home() / ".freesurfer_license"
LICENSE_PATH.write_text(LICENSE_TEXT)

# Refuse to continue if the placeholder is still in place — otherwise fmriprep
# runs for ~15 s building the workflow before hitting the same error.
if "your.email@example.org" in LICENSE_TEXT or "XXXXX" in LICENSE_TEXT:
    raise RuntimeError(
        "FreeSurfer license is still the placeholder. "
        "Paste your real license into LICENSE_TEXT above and re-run this cell."
    )

print(f"Wrote FreeSurfer license to {LICENSE_PATH}")

## 3. Paths and environment

We mirror the environment variables from the Neurodesk tutorial:

- `SUBJECTS_DIR` / `APPTAINERENV_SUBJECTS_DIR` — where FreeSurfer writes
  its per-subject directories. The `APPTAINERENV_` prefix propagates
  the variable into the Apptainer container that Neurodesk launches.
- `ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS` — caps parallelism in the
  ITK-based registration steps (also used below as `--nprocs`).
- `MPLCONFIGDIR` — matplotlib cache, kept outside `/tmp` to avoid
  permission issues inside the container.

In [ ]:
import os

PROJECT_DIR = Path.home() / "Desktop" / "fmri_course_JLU"
BIDS_DIR = PROJECT_DIR / "bids"
OUT_DIR = BIDS_DIR / "derivatives" / "fmriprep-output"
WORK_DIR = PROJECT_DIR / "fmriprep-work"
FS_SUBJECTS_DIR = BIDS_DIR / "derivatives" / "freesurfer"

for d in (OUT_DIR, WORK_DIR, FS_SUBJECTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

os.environ["SUBJECTS_DIR"] = str(FS_SUBJECTS_DIR)
os.environ["APPTAINERENV_SUBJECTS_DIR"] = str(FS_SUBJECTS_DIR)
os.environ["ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS"] = "2"
os.environ["MPLCONFIGDIR"] = str(Path.home() / "matplotlib-mpldir")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

PARTICIPANT = "2604201"
assert (BIDS_DIR / f"sub-{PARTICIPANT}").exists(), (
    f"expected {BIDS_DIR}/sub-{PARTICIPANT} to exist — run bids_conversion.ipynb first"
)

print("BIDS_DIR         :", BIDS_DIR)
print("OUT_DIR          :", OUT_DIR)
print("WORK_DIR         :", WORK_DIR)
print("FS_SUBJECTS_DIR  :", FS_SUBJECTS_DIR)
print("PARTICIPANT      :", PARTICIPANT)

## 4. Run fMRIPrep

This is a long-running job (typically **1–3 hours** for one subject with
FreeSurfer reconstruction on a workstation). Log output is streamed into
the notebook; you can also follow it in `fmriprep-output/logs/` once
fMRIPrep starts writing there.

Flags (matches the Neurodesk tutorial):

| Flag | Reason |
|---|---|
| `--fs-license-file` | FreeSurfer license |
| `--output-spaces T1w MNI152NLin2009cAsym fsaverage fsnative` | common BIDS-Derivatives spaces |
| `--participant-label 2604201` | single-subject run |
| `--nprocs 2 --mem 10000` | resource caps (adjust for your machine) |
| `--skip_bids_validation` | dataset has extra fmap files from dcm2niix that the validator complains about; preprocessing itself is unaffected |
| `--fs-subjects-dir` | reuse FreeSurfer outputs across re-runs |
| `-w fmriprep-work` | scratch dir (can be deleted after) |
| `-v` | verbose logging |

In [ ]:
### Reset stale state from a previous interrupted fMRIPrep run.
#
# Run this cell *only* if a previous fmriprep run crashed and left:
#   - `IsRunning.lh` / `IsRunning.rh` lockfiles in the FreeSurfer subject dir
#     (FreeSurfer error: "subject folder contains IsRunning files...")
#   - a half-written work dir that fmriprep then complains about with
#     `FileNotFoundError: anat_to_template_Warped.nii.gz`
#
# It removes the FS lockfiles and the per-subject anat work dir, but keeps
# any successful recon-all output so you don't have to redo the slow part.

import shutil

fs_subj = FS_SUBJECTS_DIR / f"sub-{PARTICIPANT}"
for lock in fs_subj.glob("scripts/IsRunning.*"):
    print("removing", lock)
    lock.unlink()

anat_wf = WORK_DIR / "fmriprep_24_1_wf" / f"sub_{PARTICIPANT}_wf" / "anat_fit_wf"
if anat_wf.exists():
    print("removing", anat_wf)
    shutil.rmtree(anat_wf)

print("\nreset complete — re-run the next cell")

In [ ]:
import subprocess

# If you hit `subprocess exited with code 137` (= the kernel OOM-killed a
# FreeSurfer surface tool) on a memory-constrained machine, set this to True.
# Lightweight mode skips FreeSurfer surface reconstruction and the surface
# output spaces (`fsaverage`, `fsnative`). You still get full volumetric
# preprocessing + MNI normalisation, which is enough for most undergrad
# analyses. Saves ~12 GB of peak RAM.
LIGHTWEIGHT = True

cmd = [
    "fmriprep",
    str(BIDS_DIR),
    str(OUT_DIR),
    "participant",
    "--fs-license-file", str(LICENSE_PATH),
    "--participant-label", PARTICIPANT,
    "--nprocs", os.environ["ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS"],
    "--mem", "20000",
    "--skip_bids_validation",
    "--fs-subjects-dir", str(FS_SUBJECTS_DIR),
    "-w", str(WORK_DIR),
    "-v",
]

if LIGHTWEIGHT:
    cmd += [
        "--fs-no-reconall",
        "--output-spaces", "T1w", "MNI152NLin2009cAsym",
    ]
else:
    cmd += [
        "--output-spaces", "T1w", "MNI152NLin2009cAsym", "fsaverage", "fsnative",
    ]

print(">>>", " ".join(cmd), "\n")

proc = subprocess.run(cmd)
print("\nreturncode:", proc.returncode)

## 5. Inspect the outputs

When fMRIPrep finishes, the main artefacts are:

- `fmriprep-output/sub-2604201.html` — per-subject QC report
- `fmriprep-output/sub-2604201/anat/` — preprocessed T1w, brain masks,
  tissue segmentations, transforms to each `--output-spaces` target
- `fmriprep-output/sub-2604201/func/` — motion-corrected and resampled
  BOLD plus `confounds_timeseries.tsv`
- `fmriprep-freesurfer-dir/sub-2604201/` — FreeSurfer `recon-all` output

In [ ]:
if OUT_DIR.exists():
    print("fmriprep-output/")
    for p in sorted(OUT_DIR.rglob("*")):
        depth = len(p.relative_to(OUT_DIR).parts)
        if depth <= 3:
            print("  " * depth + p.name)